<a href="https://colab.research.google.com/github/sequze/KR_BD_10_00/blob/master/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Парная регрессия
Модель линейной регрессии строится в виде: $\widehat Y = b_0 + b_1 X$
коэффициены $b_0$ и $b_1$ можно найти следующим образом:

$$b_1 = \frac{\overline{xy} - \bar{x}\bar{y}}{\overline{x^2} - {\bar{x}}^2}, b_0 =  \bar{y} - b_1 \bar{x}, $$
где
$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i, \quad \overline{x^2} = \frac{1}{n}\sum_{i=1}^{n} {x_i}^2, \quad \bar{y} = \frac{1}{n}\sum_{i=1}^{n} y_i, \quad \overline{xy} = \frac{1}{n}\sum_{i=1}^{n} x_i y_i $$

Скачаем файл с данными demad.csv

In [ ]:
!pip install wldhx.yadisk-direct

In [ ]:
!curl -L $(yadisk-direct https://disk.yandex.ru/d/jRi45rI4l2XlpA) -o demand.csv

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('demand.csv', sep=';')
df

In [ ]:
def mnk(X, Y):
  x_ = X.sum() / len(X)
  y_ = Y.sum() / len(Y)
  x2_ = sum(i ** 2 for i in X) / len(X)
  # zip - итератор по набору данных
  xy_ = sum(i * j for i,j in zip(X,Y)) / len(X)

  b_1 = (xy_ - x_ * y_) / (x2_ - x_ * x_)
  b_0 = y_ - b_1 * x_

  return (b_0, b_1)

In [ ]:
(b_0, b_1) = mnk(df['P'], df['Q'])
print(b_0, b_1)

99.94767199666741 -1.953087448328901


Итоговое уравнение: $ Q = 99.95 - 1.95 P + e$

Оценим параметры линейной регрессии с помощью библиотеки _sklearn_

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [ ]:
model = LinearRegression()
lst = [[df['P'][i]] for i in range(len(df['P']))]
#print(lst)
#print(df['P'])
model.fit(lst,df['Q'])
print(model. intercept_ , model. coef_ , model. score (lst,df['Q']))


99.94767199666741 [-1.95308745] 0.9862525267244945


С сайта finam.ru скачайте цены закрытия какого-нибудь финансового инструмента (не менее 500 наблюдений). Методом наименьших квадратов постройте линию тренда, откладывая по оси OX номер момента времени, по оси OY — цену закрытия

## Линия тренда для цен закрытия акций Роснефти

Источник данных: [экспорт котировок Роснефти на Finam](https://www.finam.ru/quote/moex/rosn/export/).  
Периодичность — **1 день**, период в исходной выгрузке — **12.07.2024–14.09.2026**. После обработки осталось 675 наблюдений и два столбца: порядковый номер момента времени `NUMBER` и цена закрытия `CLOSE`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

data_url = 'https://raw.githubusercontent.com/sequze/KR_BD_10_00/master/rosneft_close.csv'
rosn = pd.read_csv(data_url, sep=';')

assert list(rosn.columns) == ['NUMBER', 'CLOSE']
assert len(rosn) >= 500
rosn

In [ ]:
X = rosn['NUMBER']
Y = rosn['CLOSE']
b_0, b_1 = mnk(X, Y)
rosn['TREND'] = b_0 + b_1 * X

print(f'Количество наблюдений: {len(rosn)}')
print(f'b₀ = {b_0:.6f}')
print(f'b₁ = {b_1:.6f}')
print(f'Уравнение тренда: CLOSE = {b_0:.4f} {b_1:+.6f} · NUMBER')

In [ ]:
model = LinearRegression()
model.fit(rosn[['NUMBER']], rosn['CLOSE'])

print(f'Коэффициенты sklearn: b₀ = {model.intercept_:.6f}, b₁ = {model.coef_[0]:.6f}')
print(f'Коэффициент детерминации R² = {model.score(rosn[["NUMBER"]], rosn["CLOSE"]):.6f}')

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(rosn['NUMBER'], rosn['CLOSE'], color='steelblue', linewidth=1.2, label='Цена закрытия')
plt.plot(rosn['NUMBER'], rosn['TREND'], color='crimson', linewidth=2.5, label='Линия тренда (МНК)')
plt.xlabel('Номер момента времени')
plt.ylabel('Цена закрытия, руб.')
plt.title('Дневные цены закрытия акций Роснефти и линейный тренд')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

### Вывод

Получена линия тренда:

$$\widehat{CLOSE} = 525.8448 - 0.259416 \cdot NUMBER.$$

Коэффициент наклона отрицательный: в рамках рассматриваемого периода линейная модель показывает среднее снижение цены закрытия примерно на **0.259 рубля за одно торговое наблюдение**. Коэффициент детерминации $R^2 \approx 0.6136$, то есть линейный тренд объясняет около 61.36% вариации цены в этой выборке.